In [0]:
import pandas as pd
RAW = "/Volumes/workspace/raw/files"
read = lambda f, skip: pd.read_csv(f"{RAW}/{f}", skiprows=skip, dtype=str,encoding="utf-8-sig", keep_default_na=False)
ica = read("ica.csv", 3)              # 3 blank rows above the header 
de  = read("disaster_events.csv", 0)  # header is the first row 
pop = read("abs_population.csv", 6)   # 6 title/metadata rows (including #REF!) 
inc = read("abs_income.csv", 6) 
for name, df in [("ica", ica), ("de", de), ("pop", pop), ("inc", inc)]:
    print(name, df.shape)

In [0]:
print("1. Events vs rows, and undeclared events") 
print("Total row count:",len(ica),"\nUnique category names: ", ica["CAT Name"].nunique(),"\nUndeclared count: ",(ica["CAT Name"] == "Undeclared").sum())   
print("2. State values: expect codes, sub-state regions (FNQ, SEQ), mixed case, lists, blanks")
print(ica["State"].value_counts(dropna=False))
print("2b. State blanks for events since 2011")
d = pd.to_datetime(ica["Event Start"], format="%d-%b-%y", errors="coerce")
d = d.where(d.dt.year <= 2026, d - pd.DateOffset(years=100))
since = d.dt.year >= 2011
print(since.sum(), "events since 2011")
print((ica.loc[since, "State"].str.strip() == "").sum(), "have a blank state")
print("3. Two-digit years: how many dates parse into the future?") 
d = pd.to_datetime(ica["Event Start"], format="%d-%b-%y", errors="coerce") 
print((d.dt.year > 2026).sum(), "dates land in 2027 or later")
print("4. Placeholders in number columns")
print((ica["Domestic Building Claims"].str.strip() == "-").sum(), "dash placeholders")
print((ica["TOTAL CLAIMS RECEIVED"].str.strip() == "").sum(), "blank claim counts")
print("5. Disaster events: two date formats, coverage ends in 2014")
for fmt in ["%m/%d/%Y %H:%M", "%d/%m/%Y %H:%M:%S"]:    
    print(fmt, pd.to_datetime(de["startDate"], format=fmt, errors="coerce").notna().sum()) 
print("6a. ABS: which years actually have values?")
st = pop[pop["Code"].str.fullmatch(r"[1-8]")] 
print(st.groupby("Year")["Estimated resident population (no.)"].apply(lambda s: (s != "-").sum()))
print("6b. ABS income: which years actually have values?")
ic = "Median total income (excl. Government pensions and allowances) ($)"
st = inc[inc["Code"].str.fullmatch(r"[1-8]")]
print(st.groupby("Year")[ic].apply(lambda s: (s != "-").sum()))
print("7. Out-of-scope events: man-made hazards and outside Australia")
print(ica["Type"].value_counts())
oos = ica.loc[ica["Type"] == "Man-made", ["Event Name", "State", "Year"]].copy()
oos.index = oos.index + 1          # 1-based, matches _row_num in staging
oos.index.name = "Source row"      # heading for the row numbers
print(oos)